# Trabajo Práctico Aprendizaje Automático 2 - Predicción de puntuación y capitalización en texto normalizado

In [1]:
import os
import re
from datasets import load_dataset
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import joblib
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import unicodedata


from dataclasses import dataclass

In [2]:
# Seteamos la semilla para que los resultados sean reproducibles
SEED = 2026
np.random.seed(SEED)

rng = np.random.default_rng(SEED)

In [3]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

In [4]:
def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)
    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌ El token '{token}' no pertenece al vocabulario de multilingualBERT.")
        return None
    embedding_vector = model.embeddings.word_embeddings.weight[token_id]

    return embedding_vector

## Datasets

Combinamos dos fuentes de datos en español (ambas se leen en streaming desde HuggingFace):

1. **Parlamento Europeo** (`Helsinki-NLP/europarl`): texto formal, actas parlamentarias. Las oraciones vienen en el orden original del discurso. Fuente: [Europarl](https://statmt.org/europarl/)
2. **OPUS-100** (`Helsinki-NLP/opus-100`): mezcla de varios corpus de OPUS (subtítulos, documentos técnicos, etc.), con mucho lenguaje conversacional. Las oraciones vienen mezcladas, así que dos oraciones consecutivas no tienen relación entre sí. Aporta la mayoría de las preguntas.

### Limpieza de oraciones

Cada oración pasa por tres pasos:
1. **Descarte directo** de casos donde el `.` no marca fin de oración (abreviaturas como `Sr.`, iniciales, números como `1.500`), puntos suspensivos y URLs.
2. **Mapeo de signos** que no predecimos a los que sí: `!` → `.`, `;` y `:` → `,`. Se eliminan `¡`, comillas, paréntesis, y guiones/apóstrofes que no estén dentro de una palabra.
3. **Validación**: largo razonable, solo caracteres permitidos, empieza en mayúscula, termina en `.` o `?` y tiene los `¿?` balanceados.

In [5]:
# Parámetros del dataset
N_ORACIONES_POR_FUENTE = 5000        # Oraciones limpias a leer de cada fuente
MIN_PALABRAS_ORACION = 1             # Descartamos oraciones más cortas
MAX_PALABRAS_ORACION = 60            # Descartamos oraciones más largas
MAX_ORACIONES_POR_DOCUMENTO = 20     # Oraciones consecutivas que forman un "documento"

MIN_PALABRAS_INSTANCIA = 10          # Largo mínimo de cada recorte
MAX_PALABRAS_INSTANCIA = 50          # Largo máximo de cada recorte
PROB_ORACIONES_SUELTAS = 0.15        # Proporción de documentos que se usan como oraciones sueltas

PROPORCION_TEST = 0.2              # El resto queda para train

RUTA_DATOS = "datos"                 # Carpeta donde se guarda el dataset ya armado
RUTA_MODELOS = "modelos"             # Carpeta donde se guardan los modelos entrenados

In [6]:
ABREVIATURAS = {
    "sr.", "sra.", "sres.", "sras.", "srta.", "dr.", "dra.", "ud.", "uds.",
    "art.", "arts.", "núm.", "nº.", "pág.", "págs.", "vol.", "cap.",
    "etc.", "ej.", "p.ej.", "aprox.", "ee.uu.", "a.c.", "d.c.",
}

CARACTERES_PERMITIDOS = set(" ¿?.,'-")

# Decidir si descartar de entrada o si se puede limpiar de alguna forma
def descartar_de_entrada(texto):
    """Casos que no vale la pena intentar limpiar."""
    if "..." in texto or "…" in texto:
        return True
    if "http" in texto or "www" in texto or "@" in texto:
        return True
    # Números con punto o coma (ej: "1.500", "2,5"): el "." no sería fin de oración
    if re.search(r"\d[.,]\d", texto):
        return True
    # Abreviaturas (ej: "Sr. López"): el "." no sería fin de oración
    if any(palabra.lower() in ABREVIATURAS for palabra in texto.split()):
        return True
    # Iniciales (ej: "J. M. Barroso")
    if re.search(r"\b[A-ZÁÉÍÓÚÑ]\.\s", texto):
        return True
    return False


def normalizar_espacios_puntuacion(texto):
    # Quita espacios antes de '?', '.' o ',' (ej: "texto ?" -> "texto?")
    texto = re.sub(r"\s+([?.,])", r"\1", texto)
    # Agrega un espacio después de '?', '.' o ',' si quedaron pegados (ej: "tarde,hubo")
    texto = re.sub(r"([?.,])(?=\w)", r"\1 ", texto)
    # Quita espacios después de '¿' y agrega uno antes si quedó pegado (ej: "bien,¿no?")
    texto = re.sub(r"¿\s+", "¿", texto)
    texto = re.sub(r"(?<=\S)¿", " ¿", texto)
    # Elimina espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def es_oracion_valida(texto):
    palabras = texto.split()
    if not (MIN_PALABRAS_ORACION <= len(palabras) <= MAX_PALABRAS_ORACION):
        return False

    if any(not (c.isalnum() or c in CARACTERES_PERMITIDOS) for c in texto):
        return False

    # Cada palabra: un '¿' opcional al inicio, la palabra, y a lo sumo un signo al final
    if any(not re.fullmatch(r"¿?[\w'-]+[.,?]?", palabra) for palabra in palabras):
        return False

    # Tiene que parecer una oración completa: empieza en mayúscula y termina en . o ?
    primera_letra = next((c for c in texto if c.isalpha()), "")
    if not primera_letra.isupper():
        return False
    if not (texto.endswith(".") or texto.endswith("?")):
        return False

    # Los '¿' y '?' tienen que estar balanceados y sin anidar
    balance = 0
    for char in texto:
        if char == "¿":
            balance += 1
        elif char == "?":
            balance -= 1
        if balance < 0 or balance > 1:
            return False
    if balance != 0:
        return False

    n_letras = sum(1 for c in texto if c.isalpha())
    if n_letras < len(texto) * 0.5:
        return False

    return True


def limpiar_oracion(texto):
    """Devuelve la oración limpia, o None si hay que descartarla."""
    texto = texto.strip()
    if descartar_de_entrada(texto):
        return None
    texto = normalizar_espacios_puntuacion(texto)
    if not es_oracion_valida(texto):
        return None
    return texto

In [7]:
ejemplos = [
    "Fueron los asbestos aquí. ¡Eso es lo que ocurrió!",
    "Y prefiero mudarme, ¿Entiendes?",
    "- Gracias.",
    "Sería -me parece a mí- lo mínimo, al alba del tercer milenio.",
    "Esta tarde,hubo conciertos en las ciudades ocupadas.",
    "Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.",
    "Vamos a McDonald's con el Sr. Pérez.",
    "en el apartado 2, el párrafo primero se sustituye por el texto siguiente:",
    "Perdón, he tenido una burbuja de gas extraño un momentito",
]

for ejemplo in ejemplos:
    print(f"{ejemplo!r}\n  -> {limpiar_oracion(ejemplo)!r}")

'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!'
  -> None
'Y prefiero mudarme, ¿Entiendes?'
  -> 'Y prefiero mudarme, ¿Entiendes?'
'- Gracias.'
  -> '- Gracias.'
'Sería -me parece a mí- lo mínimo, al alba del tercer milenio.'
  -> 'Sería -me parece a mí- lo mínimo, al alba del tercer milenio.'
'Esta tarde,hubo conciertos en las ciudades ocupadas.'
  -> 'Esta tarde, hubo conciertos en las ciudades ocupadas.'
'Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.'
  -> None
"Vamos a McDonald's con el Sr. Pérez."
  -> None
'en el apartado 2, el párrafo primero se sustituye por el texto siguiente:'
  -> None
'Perdón, he tenido una burbuja de gas extraño un momentito'
  -> None


In [8]:
def leer_documentos(nombre_dataset):
    """
    Lee el dataset en streaming y lo agrupa en documentos: listas de oraciones
    limpias que aparecen de forma consecutiva en la fuente.

    Si una oración se descarta, se corta el documento actual para no unir
    oraciones que en el original tenían texto en el medio.
    """
    stream = load_dataset(nombre_dataset, "en-es", split="train", streaming=True)

    documentos = []
    documento_actual = []
    n_oraciones = 0

    for ejemplo in stream:
        oracion = limpiar_oracion(ejemplo["translation"]["es"])

        if oracion is None:
            if documento_actual:
                documentos.append(documento_actual)
                documento_actual = []
            continue

        documento_actual.append(oracion)
        n_oraciones += 1

        if len(documento_actual) == MAX_ORACIONES_POR_DOCUMENTO:
            documentos.append(documento_actual)
            documento_actual = []

        if n_oraciones >= N_ORACIONES_POR_FUENTE:
            break

    if documento_actual:
        documentos.append(documento_actual)

    return documentos

### Armado de instancias

Ambos datasets están segmentados por oración. Si entrenáramos con oraciones completas, los modelos aprenderían un **sesgo posicional**: que la primera palabra siempre va en mayúscula y que `.` y `?` aparecen solo en la última palabra. Pero el enunciado aclara que cada instancia de evaluación puede contener *parte* de una o varias oraciones.

Para evitarlo:
1. **Documentos**: agrupamos hasta `MAX_ORACIONES_POR_DOCUMENTO` oraciones consecutivas válidas. Si una oración se descarta, el documento se corta, para no unir oraciones que en el original tenían texto en el medio. En *Europarl* esto reconstruye fragmentos reales de un mismo discurso. En *OPUS-100* las oraciones no tienen relación entre sí, pero como los modelos usan sobre todo contexto local, el cambio de tema entre oraciones no debería afectar mucho.
2. **Split por documento**: separamos train / test a nivel documento y *antes* de recortar. Así no hay fragmentos de un mismo texto en los dos splits. Cada instancia guarda su `documento_id`: al hacer CV sobre train hay que agrupar por esa columna (ej: `GroupKFold`) para que la misma regla se cumpla entre folds.
3. **Recortes**: juntamos las palabras de cada documento y las cortamos en tramos de largo aleatorio (entre `MIN_PALABRAS_INSTANCIA` y `MAX_PALABRAS_INSTANCIA`). Los tramos pueden empezar y terminar a mitad de oración. Una proporción `PROB_ORACIONES_SUELTAS` de los documentos se usa como oraciones sueltas.
4. **Fuentes separadas**: nunca se mezclan oraciones de *Europarl* y *OPUS-100* en una misma instancia.

### Etiquetas (a nivel palabra)

Por ahora trabajamos con palabras, no con tokens de BERT. Cada palabra tiene:
- `punt_inicial`: `¿` o `""`
- `punt_final`: `,`, `.`, `?` o `""`
- `capitalizacion`: 0 (minúsculas), 1 (primera letra), 2 (mixta, ej: `iPhone`), 3 (todo mayúsculas, más de una letra)

`palabra` es la entrada del modelo (minúsculas y sin signos). `palabra_original` conserva las mayúsculas, que sirve para la clase 2 (ej: armar un diccionario de palabras mixtas vistas en entrenamiento).

In [9]:
def recortar_instancias(documento, rng):
    """
    Convierte un documento en instancias (listas de palabras con puntuación y mayúsculas).

    - Con probabilidad PROB_ORACIONES_SUELTAS, cada oración es una instancia.
    - Si no, se juntan todas las oraciones y se recortan tramos de largo aleatorio,
    que pueden empezar y terminar a mitad de oración.
    """
    if rng.random() < PROB_ORACIONES_SUELTAS:
    # Asumimos que cada elemento de 'documento' es una oración en string o ya una lista
        return [oracion.split() if isinstance(oracion, str) else oracion for oracion in documento]


    # Si 'documento' es una lista de oraciones (strings), primero las aplanamos correctamente a nivel de palabras
    palabras = []
    for oracion in documento:
        if isinstance(oracion, str):
            palabras.extend(oracion.split())
        else:
            palabras.extend(oracion)

    instancias = []
    inicio = 0
    while inicio < len(palabras):
        largo = rng.integers(MIN_PALABRAS_INSTANCIA, MAX_PALABRAS_INSTANCIA + 1)
        instancia = palabras[inicio:inicio + largo]

        # El último tramo puede quedar muy corto: lo descartamos
        if len(instancia) >= MIN_PALABRAS_ORACION:
            instancias.append(instancia)
        inicio += largo

    return instancias



def capitalizacion(palabra):
    """0: minúsculas, 1: primera mayúscula, 2: mixta (ej: iPhone), 3: todo mayúsculas."""
    letras = [c for c in palabra if c.isalpha()]
    if not any(c.isupper() for c in letras):
        return 0
    if len(letras) > 1 and all(c.isupper() for c in letras):
        return 3
    if letras[0].isupper() and not any(c.isupper() for c in letras[1:]):
        return 1
    return 2


def etiquetar_palabra(palabra_cruda):
    """
    Recibe una palabra tal como aparece en el texto (ej: "¿Cuándo", "UBA?")
    y devuelve la palabra normalizada junto con sus etiquetas.
    """
    def char_a_int(char):
        if char == ".":
            return 1
        elif char == ",":
            return 2
        else:
            return 3
    # Puntuación inicial → 0: nada, 1: '¿'
    # Puntuación final → 0: nada, 1: '.', 2: ',', 3: '?'
    punt_inicial = 1 if palabra_cruda.startswith("¿") else 0
    ultimo_char = palabra_cruda[-1]
    punt_final = char_a_int(ultimo_char) if ultimo_char in ".,?" else 0
    palabra_original = palabra_cruda.strip("¿?.,")

    return {
        "palabra": palabra_original.lower(),
        "palabra_original": palabra_original,
        "punt_inicial": punt_inicial,
        "punt_final": punt_final,
        "capitalizacion": capitalizacion(palabra_original),
    }


def construir_dataset(documentos_por_fuente, rng):
    """
    Devuelve dos DataFrames:
    - df_instancias: una fila por instancia, con el texto original y el documento del que salió.
    - df_palabras: una fila por palabra, con las etiquetas (formato del TP, a nivel palabra).
    """
    filas_instancias = []
    filas_palabras = []
    instancia_id = 0
    documento_id = 0

    for fuente, documentos in documentos_por_fuente.items():
        for documento in documentos:
            for instancia in recortar_instancias(documento, rng):
                filas_instancias.append({
                    "instancia_id": instancia_id,
                    "documento_id": documento_id,
                    "fuente": fuente,
                    "texto": " ".join(instancia),
                })
                for posicion, palabra_cruda in enumerate(instancia):
                    fila = {"instancia_id": instancia_id, "posicion": posicion}
                    fila.update(etiquetar_palabra(palabra_cruda))
                    filas_palabras.append(fila)
                instancia_id += 1
            documento_id += 1

    df_instancias = pd.DataFrame(filas_instancias)
    df_palabras = pd.DataFrame(filas_palabras)
    df_palabras = df_palabras.merge(df_instancias[["instancia_id", "documento_id", "fuente"]], on="instancia_id")
    return df_instancias, df_palabras

In [10]:
ruta_instancias = os.path.join(RUTA_DATOS, "instancias.csv")
ruta_palabras = os.path.join(RUTA_DATOS, "palabras.csv")

# ACÁ SE DEFINEN df_instancias y df_palabras, ya sea cargándolos desde disco o armándolos desde cero
if os.path.exists(ruta_instancias) and os.path.exists(ruta_palabras) and False: # Si queres cargar desde disco, cambia False por True
    # keep_default_na=False para que la "sin puntuación" ("") no se lea como NaN
    df_instancias = pd.read_csv(ruta_instancias, keep_default_na=False)
    df_palabras = pd.read_csv(ruta_palabras, keep_default_na=False)
    print("Dataset cargado desde disco")
else:
    documentos_por_fuente = {
        "europarl": leer_documentos("Helsinki-NLP/europarl"),
        "opus100": leer_documentos("Helsinki-NLP/opus-100"),
    }
    for fuente, documentos in documentos_por_fuente.items():
        print(f"{fuente}: {len(documentos)} documentos")

    df_instancias, df_palabras = construir_dataset(documentos_por_fuente, rng)

    os.makedirs(RUTA_DATOS, exist_ok=True)
    df_instancias.to_csv(ruta_instancias, index=False)
    df_palabras.to_csv(ruta_palabras, index=False)
    print("Dataset armado y guardado en", RUTA_DATOS)

print(f"Instancias: {len(df_instancias)} | Palabras: {len(df_palabras)}")

europarl: 966 documentos
opus100: 1608 documentos
Dataset armado y guardado en datos
Instancias: 7275 | Palabras: 157444


### Chequeo de clases

In [11]:
# Distribución de clases a nivel palabra (que es como se evalúa), por fuente
for columna in ["punt_inicial", "punt_final", "capitalizacion"]:
    tabla = pd.crosstab(df_palabras[columna], df_palabras["fuente"], margins=True)
    porcentajes = pd.crosstab(df_palabras[columna], df_palabras["fuente"], normalize="columns").round(4) * 100
    print(f"--- {columna} (cantidad de palabras) ---")
    print(tabla)
    print(f"--- {columna} (% por fuente) ---")
    print(porcentajes, "\n")

--- punt_inicial (cantidad de palabras) ---
fuente        europarl  opus100     All
punt_inicial                           
0               116638    39645  156283
1                  200      961    1161
All             116838    40606  157444
--- punt_inicial (% por fuente) ---
fuente        europarl  opus100
punt_inicial                   
0                99.83    97.63
1                 0.17     2.37 

--- punt_final (cantidad de palabras) ---
fuente      europarl  opus100     All
punt_final                           
0             105242    33241  138483
1               4890     4253    9143
2               6506     2151    8657
3                200      961    1161
All           116838    40606  157444
--- punt_final (% por fuente) ---
fuente      europarl  opus100
punt_final                   
0              90.08    81.86
1               4.19    10.47
2               5.57     5.30
3               0.17     2.37 

--- capitalizacion (cantidad de palabras) ---
fuente          euro

In [12]:
# Chequeo del sesgo posicional: ¿los signos de cierre siguen apareciendo solo al final?
ultima_posicion = df_palabras.groupby("instancia_id")["posicion"].transform("max")
es_ultima = df_palabras["posicion"] == ultima_posicion
es_primera = df_palabras["posicion"] == 0

for signo in [".", "?"]:
    con_signo = df_palabras["punt_final"] == signo
    intermedios = (con_signo & ~es_ultima).sum()
    print(f"'{signo}': {con_signo.sum()} en total, {intermedios} en el medio de una instancia "
          f"({intermedios / con_signo.sum():.1%})")

termina_cortada = es_ultima & ~df_palabras["punt_final"].isin([".", "?"])
empieza_minuscula = es_primera & (df_palabras["capitalizacion"] == 0)
print(f"Instancias que terminan a mitad de oración: {termina_cortada.sum() / len(df_instancias):.1%}")
print(f"Instancias que empiezan en minúscula: {empieza_minuscula.sum() / len(df_instancias):.1%}")

'.': 0 en total, 0 en el medio de una instancia (nan%)
'?': 0 en total, 0 en el medio de una instancia (nan%)
Instancias que terminan a mitad de oración: 100.0%
Instancias que empiezan en minúscula: 48.3%


/tmp/ipykernel_23750/3473395508.py:10: RuntimeWarning: invalid value encountered in scalar divide
  f"({intermedios / con_signo.sum():.1%})")


# Herramientas comunes a todos los modelos

Todo lo de esta sección sirve para cualquier modelo (clásicos con o sin embeddings, RNN, etc.), siempre que se escriban dos funciones:

- `entrenar(df_train) -> modelos`: recibe palabras etiquetadas (como `df_palabras`) y devuelve lo que haga falta para predecir.
- `predecir(modelos, df) -> DataFrame`: recibe palabras normalizadas (columnas `instancia_id`, `posicion`, `palabra`) y devuelve un DataFrame **con el mismo índice que `df`** y una columna por tarea (`punt_inicial`, `punt_final`, `capitalizacion`).

Con eso se pueden usar `validacion_cruzada`, `evaluar`, `etiquetar_parrafo` y `comparar_parrafo`.

Para los modelos clásicos (una predicción por palabra a partir de un vector de atributos) además está armado el pipeline completo: solo hay que pasarle la función que extrae los atributos y la que crea el clasificador.

In [13]:
TAREAS = ["punt_final", "capitalizacion", "punt_inicial"]

CLASES = {
    "punt_inicial": ["¿", ""],
    "punt_final": [",", ".", "?", ""],
    "capitalizacion": [0, 1, 2, 3],
}

# Parámetros de la validación
N_FOLDS = 1                  # 1: una sola partición train / validación. >1: k-fold
PROPORCION_VALIDACION = 0.2  # Solo se usa si N_FOLDS == 1
COLUMNA_GRUPO = "documento_id"  # Lo que nunca se separa entre train y validación: "documento_id" o "instancia_id"

# Columnas de df_palabras que no son atributos (identificadores y etiquetas)
COLUMNAS_NO_ATRIBUTOS = [
    "instancia_id", "documento_id", "fuente", "split", "palabra", "palabra_original",
    "punt_inicial", "punt_final", "capitalizacion",
]

# Función auxiliar para obtener las columnas de atributos a partir de un DataFrame
def columnas_de_atributos(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in COLUMNAS_NO_ATRIBUTOS]

In [14]:
# Dividir en conjunto de desarrollo y control

X_train, X_test, y_train, y_test = train_test_split(
    df_palabras.drop(columns=["punt_final", "capitalizacion", "punt_inicial"]),
    df_palabras[["punt_final", "capitalizacion", "punt_inicial"]],
    test_size=PROPORCION_TEST,
    random_state=SEED,
    stratify=df_palabras["fuente"]
)

df_palabras = pd.concat([X_train, y_train], axis=1)
df_test_palabras = pd.concat([X_test, y_test], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    df_instancias.drop(columns=["texto"]),
    df_instancias["texto"],
    test_size=PROPORCION_TEST,
    random_state=SEED,
    stratify=df_instancias["fuente"]
)

df_instancias = pd.concat([X_train, y_train], axis=1)
df_test_instancias = pd.concat([X_test, y_test], axis=1)

#### Funciones de reconstrucción de texto con las predicciones

A 'reconstruir_texto_predicho' le pasas un DataFrame con las predicciones por palabras y reconstruye la oración aplicando las predicciones.
EL DataFrame debe estar constituido por las columnas:

- punt_inicial_pred
- punt_final_pred
- capitalizacion_pred

In [15]:
# Funciones para reconstruir el texto a partir de las etiquetas, para poder visualizar la predicción

def map_numerito_a_punt_inical(numerito):
    if numerito == 0:
        return ""
    elif numerito == 1:
        return "¿"
def map_numerito_a_punt_final(numerito):
    if numerito == 0:
        return ""
    elif numerito == 1:
        return "."
    elif numerito == 2:
        return ","
    return "?"


def reconstruir_texto(palabras_instancia):
    """Arma el texto con puntuación y mayúsculas a partir de las etiquetas de cada palabra."""
    partes = []
    for _, fila in palabras_instancia.iterrows():
        palabra = fila["palabra"]
        if fila["capitalizacion"] == 1:
            palabra = palabra[0].upper() + palabra[1:]
        elif fila["capitalizacion"] == 2:
            palabra = fila["palabra_original"]  # no hay forma de deducir qué letras van en mayúscula PROBLEMA
        elif fila["capitalizacion"] == 3:
            palabra = palabra.upper()
        partes.append(map_numerito_a_punt_inical(fila["punt_inicial"]) + palabra + map_numerito_a_punt_final(fila["punt_final"]))
    return " ".join(partes)

def normalizar_parrafo(texto):
    """Pasa a minúsculas, saca los signos de puntuación y separa en palabras."""
    texto = re.sub(r"[¿?¡!.,;:\"«»()]", " ", texto.lower())
    return texto.split()


def texto_predicho(prediccion):
    """Reconstruye el texto a partir de las predicciones."""
    prediccion = prediccion.copy()
    prediccion["palabra_original"] = prediccion["palabra"].str.capitalize()

    return reconstruir_texto(prediccion)

def reconstruir_texto_predicho(df_predicciones):
    """Reconstruye el texto limpio usando las columnas de predicción."""
    prediccion = df_predicciones.copy()
    prediccion["punt_inicial"] = prediccion["punt_inicial_pred"]
    prediccion["punt_final"] = prediccion["punt_final_pred"]
    prediccion["capitalizacion"] = prediccion["capitalizacion_pred"]
    prediccion["palabra_original"] = prediccion["palabra"]
    return reconstruir_texto(prediccion)

### Pipeline un Modelo por Tarea

In [16]:
def entrenar(X, ys, nuevo_modelo):
    modelos = {}
    for target in ys.columns:
        modelo = nuevo_modelo()
        modelo.fit(X, ys[target])
        modelos[target] = modelo
    return modelos

def validar(X_test, ys_test, modelos):
    etiquetas = {
        "punt_final": {0: "Sin signo", 1: ".", 2: ",", 3: "?"},
        "capitalizacion": {
            0: "minúsculas",
            1: "Inicial mayúscula",
            2: "Mixta",
            3: "MAYÚSCULAS",
        },
        "punt_inicial": {0: "Sin signo", 1: "¿"},
    }
    filas = []
    for tarea in ys_test.columns:
        clases = list(range(len(CLASES[tarea])))
        y_true = ys_test[tarea]
        y_pred = modelos[tarea].predict(X_test)
        f1_por_clase = f1_score(
            y_true, y_pred, labels=clases, average=None, zero_division=0
        )

        for clase, f1 in zip(clases, f1_por_clase):
            filas.append({
                "Tarea": tarea,
                "Clase": clase,
                "Etiqueta": etiquetas.get(tarea, {}).get(clase, str(clase)),
                "F1": f1,
            })

        filas.append({
            "Tarea": tarea,
            "Clase": "Macro",
            "Etiqueta": "Promedio no ponderado",
            "F1": f1_score(
                y_true, y_pred, labels=clases, average="macro", zero_division=0
            ),
        })

    return (
        pd.DataFrame(filas)
        .style
        .format({"F1": "{:.3f}"})
        .background_gradient(cmap="YlGn", subset=["F1"], vmin=0, vmax=1)
        .set_caption("F1 por clase en el conjunto de prueba")
    )

# Random Forest

In [17]:
# Hiperparámetros de los Random Forest
PARAMS_RF = {
    "n_estimators": 100,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",  # las clases ¿ ? y la capitalización 2 y 3 son muy raras
}


def nuevo_random_forest():
    return RandomForestClassifier(**PARAMS_RF, random_state=SEED, n_jobs=-1)


## Random Forest (atributos a mano, sin embeddings)

Entrenamos un Random Forest independiente por tarea, todos con los mismos atributos de las palabras (y de sus vecinas):

```
                          ┌─► RF punt_final (, . ? "")
palabras ─► atributos ────┼─► RF capitalizacion (0 1 2 3)
                          └─► RF punt_inicial (¿ "")
```

Se crea el DataFrame de atributos y se lo pasas a la función 'entrenar' que crea un modelo por TAREA.
La función 'validar' se encarga de chequear cada modelo.

### Atributos

In [18]:
VOCALES = set("aeiouáéíóúü")
VOCALES_SIN_TILDE = set("aeiouü")

ADVERBIOS_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cómo", "cuándo", "dónde",
    "cuánto", "cuánta", "cuántos", "cuántas", "cuál", "cuáles",
}

PRONOMBRES_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cuál", "cuáles", "copauánto",
    "cuánta", "cuántos", "cuántas",
}

# Unimos ambos para el atributo "es_palabra_interrogativa"
PALABRAS_INTERROGATIVAS = ADVERBIOS_INTERROGATIVOS | PRONOMBRES_INTERROGATIVOS

# Palabras que casi nunca cierran una oración (artículos, preposiciones, posesivos)
PALABRAS_FUNCIONALES = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "lo",
    "de", "del", "a", "al", "en", "con", "por", "para", "sin", "sobre",
    "entre", "hacia", "desde", "hasta", "según",
    "mi", "tu", "su", "mis", "tus", "sus",
    "nuestro", "nuestra", "nuestros", "nuestras",
}

# Palabras que suelen abrir una cláusula nueva (antes suele ir una coma o un punto)
CONECTORES = {
    "pero", "sino", "aunque", "porque", "pues", "entonces", "además",
    "embargo", "mientras", "cuando", "donde", "como", "que", "y", "o", "ni",
}

def _es_palabra_interrogativa(palabra: str) -> int:
    return int(palabra in PALABRAS_INTERROGATIVAS)

In [19]:
# Atributos de las palabras vecinas que se agregan como contexto
COLUMNAS_CONTEXTO = [
    "es_palabra_interrogativa", "tiene_tilde", "es_funcional", "es_conector", "cantidad_letras",
]


def _tiene_tilde(palabra: str) -> int:
    """1 si la palabra contiene al menos una vocal con tilde (á é í ó ú)."""
    return int(any(ch in "áéíóú" for ch in palabra))


def _cantidad_vocales(palabra: str) -> int:
    return sum(1 for ch in palabra if ch in VOCALES)


def _cantidad_consonantes(palabra: str) -> int:
    """Cuenta letras que no son vocales. Ignora dígitos, espacios, etc."""
    return sum(
        1 for ch in palabra
        if unicodedata.category(ch).startswith("L") and ch not in VOCALES
    )


def extraer_atributos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega atributos léxicos y contextuales a un DataFrame a nivel palabra
    (como df_palabras). Conserva el índice de df.

    Solo se usan atributos que se pueden calcular COLUMNAS_NO_ATRIBUTOS = [
    "instancia_id", "documento_id", "fuente", "split", "palabra", "palabra_original",
    "punt_inicial", "punt_final", "capitalizacion",
]sobre el texto normalizado:
    nada que dependa de las etiquetas, porque en inferencia no las tenemos.
    """
    df = df.copy()

    columnas_requeridas = ["instancia_id", "posicion", "palabra"]
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en el DataFrame: {faltantes}")

    # Orden explícito para que los shifts prev/next sean correctos.
    df = df.sort_values(["instancia_id", "posicion"])

    # Atributos por palabra
    df["tiene_tilde"] = df["palabra"].map(_tiene_tilde)
    df["cantidad_vocales"] = df["palabra"].map(_cantidad_vocales)
    df["cantidad_consonantes"] = df["palabra"].map(_cantidad_consonantes)
    df["cantidad_letras"] = df["cantidad_vocales"] + df["cantidad_consonantes"]
    df["es_palabra_interrogativa"] = df["palabra"].map(_es_palabra_interrogativa)
    df["es_funcional"] = df["palabra"].isin(PALABRAS_FUNCIONALES).astype(int)
    df["es_conector"] = df["palabra"].isin(CONECTORES).astype(int)
    df["tiene_digitos"] = df["palabra"].str.contains(r"\d").astype(int)
    df["termina_en_mente"] = df["palabra"].str.endswith("mente").astype(int)
    df["termina_en_r"] = df["palabra"].str.endswith("r").astype(int)  # infinitivos

    # Posición dentro de la instancia
    df["posicion_desde_final"] = df.groupby("instancia_id")["posicion"].transform("max") - df["posicion"]

    # Contexto: atributos de las 2 palabras anteriores y las 2 siguientes dentro de la
    # misma instancia. -1 indica que no hay palabra (inicio o fin de la instancia).
    for col in COLUMNAS_CONTEXTO:
        for d in [1, 2]:
            df[f"prev{d}_{col}"] = df.groupby("instancia_id")[col].shift(d).fillna(-1)
            df[f"next{d}_{col}"] = df.groupby("instancia_id")[col].shift(-d).fillna(-1)

    return df

In [20]:
df_palabras_atributos = extraer_atributos(df_palabras)

X = df_palabras_atributos[columnas_de_atributos(df_palabras_atributos)]
y = df_palabras_atributos[TAREAS]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=PROPORCION_TEST,
    random_state=SEED,
    stratify=df_palabras_atributos["fuente"],
)

In [21]:
modelos_rf_atributos = entrenar(X_train, y_train, nuevo_random_forest)

# Guardamos los modelos
os.makedirs(RUTA_MODELOS, exist_ok=True)
for tarea, modelo in modelos_rf_atributos.items():
    ruta_modelo = os.path.join(RUTA_MODELOS, f"rf_{tarea}.joblib")
    joblib.dump(modelo, ruta_modelo)
    print(f"Modelo {tarea} guardado en {ruta_modelo}")

Modelo punt_final guardado en modelos/rf_punt_final.joblib
Modelo capitalizacion guardado en modelos/rf_capitalizacion.joblib
Modelo punt_inicial guardado en modelos/rf_punt_inicial.joblib


In [22]:
validar(X_test, y_test, modelos_rf_atributos)

,Tarea,Clase,Etiqueta,F1
0,punt_final,0,Sin signo,0.856
1,punt_final,1,.,0.357
2,punt_final,2,",",0.210
3,punt_final,3,?,0.185
4,punt_final,Macro,Promedio no ponderado,0.402
5,capitalizacion,0,minúsculas,0.903
6,capitalizacion,1,Inicial mayúscula,0.446
7,capitalizacion,2,Mixta,0.000
8,capitalizacion,3,MAYÚSCULAS,0.264
9,capitalizacion,Macro,Promedio no ponderado,0.403


In [23]:
textos_reales = [
    "¿Cómo estás? Espero que bien.",
    "Ayer fui al cine, pero no había entradas.",
    "¡Qué lástima! Mañana iré a la playa.",
]

for instancia_id, texto_real in enumerate(textos_reales):
    palabras = normalizar_parrafo(texto_real)
    df_predicciones = pd.DataFrame({
        "instancia_id": instancia_id,
        "posicion": range(len(palabras)),
        "palabra": palabras,
    })

    df_con_atributos = extraer_atributos(df_predicciones)
    columnas_atributos = columnas_de_atributos(df_con_atributos)
    X = df_con_atributos[columnas_atributos]

    for tarea in TAREAS:
        df_predicciones[f"{tarea}_pred"] = modelos_rf_atributos[tarea].predict(X)

    texto_predicho = reconstruir_texto_predicho(df_predicciones)
    print(f"Texto real {instancia_id}:     {texto_real}")
    print(f"Texto predicho {instancia_id}: {texto_predicho}")

Texto real 0:     ¿Cómo estás? Espero que bien.
Texto predicho 0: ¿Cómo estás? espero? que bien?
Texto real 1:     Ayer fui al cine, pero no había entradas.
Texto predicho 1: ¿Ayer, fui al cine, pero no había entradas?
Texto real 2:     ¡Qué lástima! Mañana iré a la playa.
Texto predicho 2: ¿Qué? lástima? mañana? iré a la Playa?


## Random Forest (con embeddings)

### Atributos

In [24]:
W = model.embeddings.word_embeddings.weight.detach().cpu().numpy()

def embedding_palabra(palabra):
    """Promedio de los embeddings estáticos de los wordpieces de la palabra."""
    ids = tokenizer.encode(str(palabra), add_special_tokens=False)
    if not ids:
        return np.zeros(W.shape[1], dtype=np.float32)
    return W[ids].mean(axis=0)

def vecinos(E, doc, paso):
    """Devuelve (ant, sig) desplazados `paso` filas."""
    ant = np.zeros_like(E)
    ant[paso:] = E[:-paso]
    ant[paso:][doc[paso:] != doc[:-paso]] = 0

    sig = np.zeros_like(E)
    sig[:-paso] = E[paso:]
    sig[:-paso][doc[:-paso] != doc[paso:]] = 0
    return ant, sig


def armar_dataframe_embeddings(df, ventana=1):
    df_nuevo = df.sort_values(["documento_id", "posicion"]).reset_index(drop=True)
    palabras = df_nuevo["palabra_original"].astype(str)

    # Un embedding por palabra única, después lookup para todas las filas
    dic = {p: embedding_palabra(p) for p in palabras.unique()}
    E = np.stack(palabras.map(dic).values).astype(np.float32)
    doc = df_nuevo["documento_id"].values
    dim = E.shape[1]  # 768

    def a_df(M, prefijo):
        return pd.DataFrame(M, columns=[f"{prefijo}_{i}" for i in range(1, dim + 1)])

    bloques = [a_df(E, "emb")]
    for paso in range(1, ventana + 1):
        ant, sig = vecinos(E, doc, paso)
        sufijo = "" if paso == 1 else str(paso)   # emb_prev_1 / emb_prev2_1 / ...
        bloques.append(a_df(ant, f"emb_prev{sufijo}"))
        bloques.append(a_df(sig, f"emb_next{sufijo}"))

    return pd.concat([df_nuevo] + bloques, axis=1)

#### Creando DataFrame

In [25]:
df_embeddings = armar_dataframe_embeddings(df_palabras, ventana=1)
print("DataFrame con embeddings generado")

DataFrame con embeddings generado


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    df_embeddings[columnas_de_atributos(df_embeddings)].drop(columns=['posicion']),
    df_embeddings[TAREAS],
    test_size=PROPORCION_TEST,
    random_state=SEED,
    stratify=df_embeddings["fuente"]
)

In [29]:
modelos_rf_emb = entrenar(X_train, y_train, nuevo_random_forest)

# Guardar los modelos entrenados en disco
os.makedirs(RUTA_MODELOS, exist_ok=True)
for tarea, modelo in modelos_rf_emb.items():
    ruta_modelo = os.path.join(RUTA_MODELOS, f"rf_{tarea}.joblib")
    joblib.dump(modelo, ruta_modelo)
    print(f"Modelo para {tarea} guardado en {ruta_modelo}")

Modelo para punt_final guardado en modelos/rf_punt_final.joblib
Modelo para capitalizacion guardado en modelos/rf_capitalizacion.joblib
Modelo para punt_inicial guardado en modelos/rf_punt_inicial.joblib


In [30]:
validar(X_test, y_test, modelos_rf_emb)

,Tarea,Clase,Etiqueta,F1
0,punt_final,0,Sin signo,0.938
1,punt_final,1,.,0.348
2,punt_final,2,",",0.256
3,punt_final,3,?,0.085
4,punt_final,Macro,Promedio no ponderado,0.407
5,capitalizacion,0,minúsculas,0.998
6,capitalizacion,1,Inicial mayúscula,0.983
7,capitalizacion,2,Mixta,0.200
8,capitalizacion,3,MAYÚSCULAS,0.991
9,capitalizacion,Macro,Promedio no ponderado,0.793
